## Open food fact

In [1]:
import polars as pl
import numpy as np
from tqdm import tqdm

import gc
gc.collect()

path_data = "../data/food.parquet"
schema = pl.read_parquet_schema(path_data)
cols = list(schema.keys())

#### colonnes


In [2]:
# selec_cols = [col for col in cols if "quality" in col.lower()]  # ingred, nutri

selec_cols = ['nutriscore_score', 'nutriscore_grade']
selec_cols += ['nutriments', 'product_name']
selec_cols += ['categories_tags', 'food_groups_tags']

# selec_cols += ['ecoscore_score', 'ecoscore_tags', 'ecoscore_data', 'nova_groups_tags', 'nova_group']
# selec_cols += ['brands_tags', 'countries_tags', 'ingredients_tags', 'entry_dates_tags']
# selec_cols += ['additives_tags', 'packaging_tags', 'ciqual_food_name_tags']
# selec_cols += ['quantity'] #pas formaté
selec_cols

['nutriscore_score',
 'nutriscore_grade',
 'nutriments',
 'product_name',
 'categories_tags',
 'food_groups_tags']

In [3]:
# del lf
lf = pl.read_parquet(path_data,
                     columns=selec_cols,
                    #  n_rows=100,
                     )
lf.shape

(3886551, 6)

In [22]:
lf = lf.with_columns([
    pl.col("product_name").list.eval(
        (
            pl.element()
                .struct.field("text")
        )
    ).list.first()
    ])

In [54]:
# ANALYSE CELL
lf.select(pl.col("product_name"))[0].item()[0]
# df.select(["product_name_text"])[0,0]
# type(lf[0]['nutriments'].item()[4])

{'lang': 'main', 'text': 'Véritable pâte à tartiner noisettes chocolat noir'}

#### nutriments

In [9]:
# ANALYSE DES NUTRIMENTS - CHOISIR LES NUTRIMENTS POUR L ANALYSE
names = (
    lf[0:400000]
    .explode("nutriments")  # On explose la liste pour avoir un dict par ligne
    .select(pl.col("nutriments").struct.field("name"))  # On extrait le champ 'name'
    .unique()  # Valeurs uniques
    .to_series()
    .to_list()
)

with open("../output/name_nutrients.txt", "w", encoding="utf-8") as f:
    for item in names:
        f.write(f"{item}\n")

In [ ]:
# EXTRACTION DES NUTRIMENTS
# ⚠️ PAS EXECUTABLE SUR L ENSEMBLE DES DONNEES

# On récupère les nutriments utilisé pour le calcul du nutri-score d'apres la doc
def extract_nutrient_value(nut_list, key):
    if nut_list is None:
        return None
    for nut in nut_list:
        name = nut.get("name", "").lower()
        if key.lower() in name:
            return nut.get("value")
    return None

# si iid feat : donc pas mettre kcal & sat-fat (?)
k = ['energy-kcal', 'fat', 'saturated-fat', 'sugars', 'proteins', 'salt', 'fiber']
k += ['fruits-vegetables-nuts']

essaie = lf[:100000].with_columns([
    pl.struct(["nutriments"])
        .map_elements(lambda x, key=key: extract_nutrient_value(x["nutriments"], key),
                      return_dtype=pl.Float64
                      ).alias(key)
                        for key in k
                                  ])

c = "food_groups_tags"
essaie = essaie.with_columns(
        pl.col(c).list.join(", ").alias(f"{c}_str")
    )

### Doublons

In [ ]:
# DOUBLONS
dups = lf.filter(
    pl.struct(["product_name", "nutriments"])
    .is_duplicated()
)
dups.shape

In [ ]:
# ANALYSE DES DOUBLONS
c='product_name'
dups_ct = dups.select(pl.col(c).value_counts()).unnest(c)
dups_ct = dups_ct.sort("count", descending=True)

dups_ct
# dups_ct.filter(pl.col("essaie").list.contains("vin"))
# dups_ct.filter(pl.col("essaie").str.contains("vin"))
# df.filter(pl.col("product_name_text_list"))

product_name,count
list[struct[2]],u32
[],239057
"[{""main"",""Miel""}, {""fr"",""Miel""}]",1035
"[{""main"",""Vin""}, {""fr"",""Vin""}]",688
"[{""main"",""Filet de poulet""}, {""fr"",""Filet de poulet""}]",554
"[{""main"",""Eier""}, {""de"",""Eier""}]",515
…,…
"[{""main"",""Kate Farms nutrition shake vanilla""}, {""en"",""Kate Farms nutrition shake vanilla""}]",2
"[{""main"",""Lietuviškas medus""}, {""lt"",""Lietuviškas medus""}]",2
"[{""main"",""Apfelmus bio""}, {""de"",""Apfelmus bio""}]",2


In [90]:
# RECHERCHE DE MOT CLE DANS UNE COLONNE DE TYPE list[struct[2]]
mot = "Vin"

df = lf.with_columns([
    pl.col("product_name").list.eval(
        (
            # pl.element()
            #     .struct.field("text")
            #     .str.to_lowercase()
            #     .str.contains(mot.lower(), literal=True)
            # | #rf"\b{mot}\b"
            (pl.element()
                .struct.field("text") 
                .str.to_lowercase() == mot.lower())
        )
    ).list.any().alias("has_vin")
])

df = df.filter(pl.col("has_vin"))
df['nutriments'] #.select(pl.col(c).value_counts()).unnest(c).sort("count", descending=True)

nutriments
list[struct[9]]
null
"[{""fiber"",0.0,0.0,null,"""",null,null,null,null}, {""salt"",0.0,0.0,null,"""",null,null,null,null}, … {""energy-kcal"",450.0,450.0,null,""kcal"",null,null,null,null}]"
"[{""alcohol"",14.5,14.5,14.5,""% vol"",null,null,null,null}, {""fruits-vegetables-legumes-estimate-from-ingredients"",null,0.0,0.0,null,null,null,null,null}, {""fruits-vegetables-nuts-estimate-from-ingredients"",null,0.0,0.0,null,null,null,null,null}]"
"[{""fruits-vegetables-nuts-estimate-from-ingredients"",null,0.0,0.0,null,null,null,null,null}, {""fruits-vegetables-legumes-estimate-from-ingredients"",null,0.0,0.0,null,null,null,null,null}, {""alcohol"",12.0,12.0,12.0,""% vol"",null,null,null,null}]"
null
…
null
null
null


In [ ]:
# TODO : doublons
# df_essaie = lf.filter(pl.col("*").is_duplicated())
lf = lf.unique(subset=["product_name", "nutriments"], keep="first")
lf.shape

# fuzzyduplicate
names = df.select("name").to_series().to_list()

from rapidfuzz import process, fuzz
# Trouver les doublons flous
matches = process.cdist(names, names, scorer=fuzz.ratio, score_cutoff=80)

(3479213, 9)

In [ ]:
# Dublons
dups = lf.filter(
    pl.struct(lf.columns)
    .is_duplicated()
)
dups.shape

In [ ]:
dups.select(pl.col(c).value_counts())
dups.filter(pl.col("product_name") == "Ketchup")
dups.filter(pl.col("product_name").str.contains("Ketchup"))